# Fine-tuning do Assistente Medico - Hospital Pos Tech

Passos: (1) instalar dependencias, (2) montar o Google Drive e subir o
projeto, (3) carregar o dataset, (4) carregar o modelo base em 4-bit,
(5) aplicar LoRA, (6) treinar com SFTTrainer, (7) testar inferencia,
(8) salvar o adaptador em `models/fine_tuned_lora/`.

In [ ]:
#Conexao com o Google Drive (rode no Colab)
from google.colab import drive
drive.mount('/content/drive')

Suba a pasta `tech_challenge` para o seu Google Drive (ex.: em
`MyDrive/tech_challenge`) e ajuste `PROJECT_DIR` abaixo.

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/tech_challenge"  # ajuste se necessario

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers peft accelerate bitsandbytes
!pip install --upgrade transformers trl datasets

!pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

DATASET_PATH = f"{PROJECT_DIR}/data/processed/dataset_fine_tuning.jsonl"
OUTPUT_MODEL_DIR = f"{PROJECT_DIR}/models/fine_tuned_lora"

max_seq_length = 2048
dtype = None
load_in_4bit = True

# Mesma familia de modelos usada no material de referencia do curso.
model_name = "unsloth/llama-3-8b-bnb-4bit"


## Carregando o dataset

O dataset ja esta no formato `instruction` / `input` / `output`
(ver `src/data_prep/build_fine_tuning_dataset.py`), o mesmo esperado pelo
prompt Alpaca usado no material de referencia.

In [ ]:
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Exemplos no dataset: {len(dataset)}")
dataset[0]

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = None  # definido apos carregar o tokenizer

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

EOS_TOKEN = tokenizer.eos_token
dataset = dataset.map(formatting_prompts_func, batched=True)

## Aplicando LoRA

Mesmos hiperparametros do material de referencia (r=16, alpha=16), que sao
um bom ponto de partida para datasets pequenos/medios como o nosso.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

In [ ]:
trainer_stats = trainer.train()

## Testando a inferencia com um caso clinico de exemplo

In [ ]:
FastLanguageModel.for_inference(model)

pergunta_teste = "Qual o protocolo interno para Diabetes Mellitus tipo 2?"

inputs = tokenizer(
    [alpaca_prompt.format(
        "Responda como assistente clinico do hospital, com base no protocolo interno.",
        pergunta_teste,
        "",
    )],
    return_tensors="pt",
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

## Bateria de perguntas de teste

Uma unica pergunta nao e suficiente para avaliar o fine-tuning. As perguntas
abaixo cobrem tres situacoes diferentes:

- **Protocolos que existem** no hospital (e portanto no dataset de
  fine-tuning) - verifica se o modelo aprendeu o conteudo clinico interno.
- **A condicao exclusiva do fine-tuning**
  (`FINE_TUNING_ONLY_CONDITIONS`, hoje "Crise Asmatica Aguda") - nunca
  esteve em `data/raw/` (RAG nao tem nada sobre ela), entao so um modelo
  que realmente aprendeu esse conteudo durante o treino consegue responder
  corretamente. Esta e a evidencia mais forte de que o fine-tuning
  aconteceu, mais forte do que qualquer pergunta que o modelo base ja
  saberia responder sozinho.
- **Um procedimento que nao existe** ("Sindrome de Kowalski-Vantablack",
  inventado para este teste) - o hospital nao tem protocolo algum sobre
  isso, nem no RAG nem no fine-tuning. O objetivo aqui nao e o modelo
  acertar uma resposta (nao ha resposta certa), e sim observar o
  *comportamento*: um modelo bem comportado deveria sinalizar que nao
  reconhece esse protocolo/condicao, em vez de inventar um protocolo
  clinico com confianca (alucinacao) - um risco real em contexto medico,
  que e exatamente por isso que o guardrail de
  `requires_human_validation`/disclaimer existe no pipeline completo
  (`src/llm/guardrails/rules.py`), fora deste notebook.

In [ ]:
FastLanguageModel.for_inference(model)

INSTRUCAO_PROTOCOLO = "Responda como assistente clinico do hospital, com base no protocolo interno."

perguntas_teste = [
    # Protocolos que existem (data/raw/protocolos_internos.json) - o modelo
    # deveria reproduzir o conteudo clinico aprendido no fine-tuning.
    ("Protocolo existente", "Qual o protocolo interno para Lombalgia Mecanica Aguda?"),
    ("Protocolo existente", "Qual o protocolo interno para Infeccao do Trato Urinario nao complicada?"),
    ("Protocolo existente", "Qual o protocolo interno para Doenca do Refluxo Gastroesofagico?"),
    # Condicao exclusiva do fine-tuning - nunca esteve em data/raw/, entao
    ("Exclusiva do fine-tuning (sem RAG)", "Qual o protocolo interno para Crise Asmatica Aguda?"),
    # Procedimento inventado, que nao existe em nenhuma fonte - testa se o modelo alucina
    ("Procedimento inexistente", "Qual o protocolo interno para a doença da coruja preguiçosa em humanos?"),
    ("Procedimento inexistente", "Qual o protocolo interno para a doença de creutzfeldt jakob?")
]

for categoria, pergunta in perguntas_teste:
    inputs = tokenizer(
        [alpaca_prompt.format(INSTRUCAO_PROTOCOLO, pergunta, "")],
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
    resposta = tokenizer.batch_decode(outputs)[0]
    print(f"=== [{categoria}] {pergunta} ===")
    print(resposta)
    print()


## Salvando o adaptador LoRA

O `src/models/domain_llm.py` carrega o modelo a partir deste caminho quando
`FINE_TUNED_MODEL_PATH` (no `.env`) apontar para ele.

In [ ]:
model.save_pretrained(OUTPUT_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)
print(f"Adaptador salvo em {OUTPUT_MODEL_DIR}")